# 🧪 Level 1 — Face Detection Dasar (OpenCV + MediaPipe)

Notebook pertama dari roadmap **Learn-faceswap**.

**Tujuan:** dari sebuah gambar, kita bisa:
1. Membaca & menampilkan gambar
2. **Mendeteksi wajah** (kotak) pakai OpenCV Haarcascade
3. Mendeteksi wajah pakai **MediaPipe** (lebih akurat)
4. **BONUS:** menampilkan 468 titik landmark wajah (Face Mesh) — persiapan ke Level 2

> 💡 Cara pakai: buka file ini di **Google Colab** (gratis). Jalankan tiap cell dari atas ke bawah (Shift + Enter).

---

## 1️⃣ Install & Import library

`opencv-python` untuk olah gambar, `mediapipe` untuk deteksi wajah/landmark, `matplotlib` untuk menampilkan hasil di Colab.

In [ ]:
# Install library (cukup sekali jalan di Colab)
!pip install -q opencv-python mediapipe matplotlib

import cv2
import mediapipe as mp
import numpy as np
import urllib.request
from matplotlib import pyplot as plt

print('OpenCV versi   :', cv2.__version__)
print('MediaPipe versi:', mp.__version__)

# Helper: tampilkan gambar OpenCV (BGR) dengan matplotlib (RGB)
def show(img, title='', size=(7, 7)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## 2️⃣ Siapkan gambar

Ada 2 opsi. Default-nya kita **download gambar contoh** otomatis supaya langsung jalan.
Kalau mau pakai foto sendiri, buka komentar di **Opsi B**.

In [ ]:
# === Opsi A: pakai gambar contoh (download otomatis) ===
img_path = 'sample_face.jpg'
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/messi5.jpg'
urllib.request.urlretrieve(url, img_path)

# === Opsi B: upload foto sendiri (hapus tanda # untuk pakai) ===
# from google.colab import files
# uploaded = files.upload()
# img_path = list(uploaded.keys())[0]

img = cv2.imread(img_path)
print('Ukuran gambar (tinggi, lebar, channel):', img.shape)
show(img, 'Gambar asli')

## 3️⃣ Face Detection — OpenCV Haarcascade (klasik)

Metode lama (2001) tapi ringan & sudah include di OpenCV. Cara kerjanya: mencari pola terang-gelap khas wajah. Kita gambar **kotak hijau** di tiap wajah yang ketemu.

In [ ]:
# Model haarcascade sudah bawaan opencv-python
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

# Deteksi bekerja di gambar grayscale (hitam-putih)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

print('Jumlah wajah terdeteksi:', len(faces))

img_haar = img.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(img_haar, (x, y), (x + w, y + h), (0, 255, 0), 3)

show(img_haar, 'Hasil Haarcascade')

## 4️⃣ Face Detection — MediaPipe (modern & lebih akurat)

MediaPipe (buatan Google) pakai jaringan saraf, jadi lebih tahan terhadap variasi pose & pencahayaan. Selain kotak wajah, ia juga kasih **6 titik kunci** (mata, hidung, telinga, mulut).

In [ ]:
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

img_mp = img.copy()
rgb = cv2.cvtColor(img_mp, cv2.COLOR_BGR2RGB)  # MediaPipe minta input RGB

with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as fd:
    results = fd.process(rgb)
    if results.detections:
        print('Jumlah wajah terdeteksi:', len(results.detections))
        for det in results.detections:
            mp_drawing.draw_detection(img_mp, det)
    else:
        print('Tidak ada wajah terdeteksi.')

show(img_mp, 'Hasil MediaPipe Face Detection')

## 5️⃣ BONUS — Face Mesh (468 titik landmark)

Ini fondasi untuk **Level 2 (landmark & alignment)** dan akhirnya **face swap**. MediaPipe Face Mesh memetakan 468 titik 3D di seluruh wajah — inilah "kerangka" yang nanti dipakai untuk menukar wajah.

In [ ]:
mp_face_mesh = mp.solutions.face_mesh
mp_styles = mp.solutions.drawing_styles

img_mesh = img.copy()
rgb = cv2.cvtColor(img_mesh, cv2.COLOR_BGR2RGB)

with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=2,
                          refine_landmarks=True, min_detection_confidence=0.5) as mesh:
    results = mesh.process(rgb)
    if results.multi_face_landmarks:
        for landmarks in results.multi_face_landmarks:
            mp_drawing.draw_landmarks(
                image=img_mesh,
                landmark_list=landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_styles.get_default_face_mesh_tesselation_style())
        print('Wajah ter-mesh:', len(results.multi_face_landmarks))
    else:
        print('Tidak ada wajah terdeteksi.')

show(img_mesh, 'Face Mesh — 468 titik landmark')

## 6️⃣ Ringkasan & Langkah Selanjutnya

Yang sudah kamu kuasai di notebook ini:
- ✅ Baca & tampilkan gambar (OpenCV + matplotlib)
- ✅ Deteksi wajah cara klasik (Haarcascade)
- ✅ Deteksi wajah cara modern (MediaPipe)
- ✅ Kenalan dengan 468 titik landmark (Face Mesh)

### 👉 Berikutnya (Level 2):
1. **Face alignment** — meluruskan wajah pakai titik landmark
2. **Face tracking** — deteksi wajah real-time di video
3. Lanjut ke **Level 4: Face Swap** pakai tool jadi (Roop / FaceFusion)

### 🧠 Coba sendiri (latihan):
- Ganti gambar di Cell 2 dengan foto yang ada **banyak wajah**, lihat bedanya.
- Ubah `minNeighbors` di Haarcascade (coba 3 vs 8), amati perubahannya.
- Bandingkan: mana yang lebih akurat, Haarcascade atau MediaPipe?